In [2]:
import numpy as np
import pandas as pd

#importation fichier .parquet
httrain= pd.read_parquet("history_toulouse_train.parquet")
httest=pd.read_parquet("history_toulouse_test.parquet")
httest

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00002 - LAFAYETTE,2026-07-31 23:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - POMME,2025-01-05 05:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PASTEUR - LAVOISIER,2025-08-05 10:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#ajout serie(colonne) heure en fomrat datetime
t=httrain.index.get_level_values("commit_at")
httrain["heure"]=t.hour
httrain["minute"]=t.minute

httrain

bikes  stands  heure  minute
station                  commit_at                                        
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9     12      15
                         2023-08-05 12:30:00     10       9     12      30
                         2023-08-05 12:45:00     10       9     12      45
                         2023-08-05 13:00:00     11       8     13       0
                         2023-08-05 13:15:00     11       8     13      15
...                                             ...     ...    ...     ...
TEST BEE                 2026-02-12 15:00:00      0       3     15       0
                         2026-02-12 15:15:00      0       3     15      15
                         2026-02-12 15:30:00      0       3     15      30
                         2026-02-12 15:45:00      0       3     15      45
                         2026-02-12 16:00:00      0       3     16       0

[40012622 rows x 4 columns]

In [5]:
#calcul moyenne des velo par quart d'heure
profil_moyen= httrain.groupby(["station", "heure", "minute"])["bikes"].mean()


profil_moyen

station                   heure  minute
00001 - POIDS DE L'HUILE  0      0         2.396884
                                 15        2.318057
                                 30        2.211732
                                 45        2.153987
                          1      0         2.055912
                                             ...   
TEST BEE                  22     45        1.000000
                          23     0         1.000000
                                 15        1.000000
                                 30        1.000000
                                 45        1.000000
Name: bikes, Length: 76528, dtype: float64

In [ ]:
# ajout des resultat dans httest
target_columns = [
    "bikes_+15min", "bikes_+30min", "bikes_+45min", "bikes_+60min",
    "bikes_+75min", "bikes_+90min", "bikes_+105min", "bikes_+120min"
]

delta_hour = [15, 30, 45, 60, 75, 90, 105, 120]


forecast= pd.to_datetime(httest.index.get_level_values("commit_at"))

for columns in range(8):
    
    time_delta= forecast + pd.Timedelta(minutes=delta_hour[columns])

    
    cles_recherche = pd.Series(list(zip(time_delta.hour, time_delta.minute)), index=httest.index)

    moyennes_futures = cles_recherche.map(meanbh)

    httest[target_columns[columns]] =  httest[target_columns[columns]].fillna(moyennes_futures).round().astype(int)


httest

In [11]:
stations=httest.index.get_level_values("station")
delta_t=httest.index.get_level_values("commit_at")

for i in range(15, 121, 15):
    cible=delta_t+pd.Timedelta(minutes=i)
    cles=pd.MultiIndex.from_arrays(
        [stations, cible.hour, cible.minute],
        names=["station", "heure", "minute"]
    )
    httest[f"bikes_+{i}min"]=profil_moyen.reindex(cles).to_numpy()

httest= httest.fillna(httrain["bikes"].mean())
httest.round().astype(int)

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,4,4,4,3,3,3,3,3
00002 - LAFAYETTE,2026-07-31 23:45:00,5,5,5,5,4,4,4,4
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,11,11,11,10,10,10,9,9
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,4,4,4,3,3,2,2,2
00003 - POMME,2025-01-05 05:45:00,8,8,8,8,8,9,9,9
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,0,0,0,0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,7,7,7,7,7,7,7,7


In [ ]:
hour_forecast= (pd.to_datetime(httest.index.get_level_values("commit_at"))).copy().hour
print(hour_forecast)
minute_forecast= (pd.to_datetime(httest.index.get_level_values("commit_at"))).copy().minute
print(minute_forecast)